# Lab 3 – NYC Taxi Data Cleaning
**Course:** INF4058 Data Engineering  
**Dataset:** `yellow_tripdata_2021-01.csv.gz`

> Follow the assignment: inspect the first 100 rows, profile the full file in chunks of 100,000 rows, count data-quality problems, document each decision, and implement `clean(df)`. Do **not** load cleaned data into PostgreSQL.

Before running, place this notebook in the same folder as `yellow_tripdata_2021-01.csv.gz`, or edit `DATA_PATH` below.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("yellow_tripdata_2021-01.csv.gz")
CHUNK_SIZE = 100_000

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Put the CSV.GZ file beside this notebook "
        "or update DATA_PATH to its full path."
    )

print("Dataset:", DATA_PATH.resolve())
print("File size (MB):", round(DATA_PATH.stat().st_size / (1024**2), 2))

## 1. Look: first 100 rows and column types
Run these cells and save screenshots of the printed output for your report.

In [ ]:
df_head = pd.read_csv(DATA_PATH, nrows=100)
print(df_head.head())
print("\\nColumn data types:")
print(df_head.dtypes)

print("\\nColumns with all values missing in these 100 rows:")
print(df_head.columns[df_head.isna().all()].tolist())

print("\\nColumns with at least one missing value in these 100 rows:")
print(df_head.columns[df_head.isna().any()].tolist())

## 2. Profile the whole file in chunks of 100,000 rows
This cell does not keep the entire dataset in memory. It aggregates null counts and min/max values across chunks. For text/object columns, min/max may be lexicographic and should be interpreted cautiously.

In [ ]:
null_counts = None
row_count = 0
dtype_seen = {}
min_values = {}
max_values = {}

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    row_count += len(chunk)

    if null_counts is None:
        null_counts = pd.Series(0, index=chunk.columns, dtype="int64")

    null_counts = null_counts.add(chunk.isna().sum(), fill_value=0).astype("int64")

    for col in chunk.columns:
        dtype_seen.setdefault(col, set()).add(str(chunk[col].dtype))
        non_null = chunk[col].dropna()
        if non_null.empty:
            continue
        try:
            chunk_min = non_null.min()
            chunk_max = non_null.max()
            if col not in min_values or chunk_min < min_values[col]:
                min_values[col] = chunk_min
            if col not in max_values or chunk_max > max_values[col]:
                max_values[col] = chunk_max
        except (TypeError, ValueError):
            # A mixed-type column may not support min/max comparison.
            pass

profile = pd.DataFrame({
    "dtype": {c: ", ".join(sorted(dtype_seen.get(c, {"unknown"}))) for c in null_counts.index},
    "null_count": null_counts,
    "null_percent": (null_counts / row_count * 100).round(3),
    "min": pd.Series(min_values),
    "max": pd.Series(max_values),
})
profile = profile.reindex(null_counts.index)
print("Total rows profiled:", row_count)
print(profile.to_string())

## 3. Find problems across the full file
The next cell counts the assignment's specified issues while continuing to read in chunks. Date columns are parsed explicitly. Duplicate rows are counted by a hash of the full row, including duplicates that occur in different chunks. Hashes are used to avoid holding the full DataFrame in memory.

In [ ]:
PICKUP_COL = "tpep_pickup_datetime"
DROPOFF_COL = "tpep_dropoff_datetime"
FARE_COL = "fare_amount"
DISTANCE_COL = "trip_distance"
PASSENGER_COL = "passenger_count"

required = [PICKUP_COL, FARE_COL, DISTANCE_COL, PASSENGER_COL]
# Check that the expected columns exist before profiling problems.
columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
missing_required = [c for c in required if c not in columns]
if missing_required:
    raise ValueError(f"Expected columns not found: {missing_required}. Available columns: {columns}")

problem_counts = {
    "pickup_outside_january_2021": 0,
    "fare_nonpositive": 0,
    "distance_nonpositive": 0,
    "passenger_count_zero": 0,
    "duplicate_rows_after_first_occurrence": 0,
}
seen_row_hashes = set()

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    pickup = pd.to_datetime(chunk[PICKUP_COL], errors="coerce")
    outside_jan = pickup.isna() | (pickup < pd.Timestamp("2021-01-01")) | (pickup >= pd.Timestamp("2021-02-01"))
    problem_counts["pickup_outside_january_2021"] += int(outside_jan.sum())

    fare = pd.to_numeric(chunk[FARE_COL], errors="coerce")
    problem_counts["fare_nonpositive"] += int((fare <= 0).sum())

    distance = pd.to_numeric(chunk[DISTANCE_COL], errors="coerce")
    problem_counts["distance_nonpositive"] += int((distance <= 0).sum())

    passenger = pd.to_numeric(chunk[PASSENGER_COL], errors="coerce")
    problem_counts["passenger_count_zero"] += int((passenger == 0).sum())

    # A duplicate means a repeated full row after its first occurrence.
    row_hashes = pd.util.hash_pandas_object(chunk, index=False).to_numpy()
    duplicate_count = 0
    for h in row_hashes:
        h = int(h)
        if h in seen_row_hashes:
            duplicate_count += 1
        else:
            seen_row_hashes.add(h)
    problem_counts["duplicate_rows_after_first_occurrence"] += duplicate_count

problem_table = pd.DataFrame(
    [{"problem": name, "count": count} for name, count in problem_counts.items()]
)
print(problem_table.to_string(index=False))

### Important note about interpretation
- The assignment asks for pickups outside January 2021. This notebook also counts unparseable pickup timestamps in that category; report this explicitly.
- Nonpositive fares/distances and zero passengers are counted exactly as requested. Decide whether to drop, fix, or flag them after reviewing the results.
- Duplicate counts are full-row duplicates after the first occurrence, not the number of all rows belonging to duplicate groups.

## 4. Document your decisions
Fill in the decision table after inspecting your counts. The sample explanations below are prompts, not facts about the dataset. Use your own words and explain who should own each rule (for example, data-quality pipeline owner or business/domain owner).

In [ ]:
decisions = pd.DataFrame([
    {
        "problem": "Pickup outside January 2021 (including unparseable pickup timestamp)",
        "decision": "DROP / FIX / FLAG",
        "why": "Write your reason based on the assignment and observed data.",
        "rule_owner": "State who should own this rule and why."
    },
    {
        "problem": "Fare amount <= 0",
        "decision": "DROP / FIX / FLAG",
        "why": "Write your reason based on the observed data.",
        "rule_owner": "State who should own this rule and why."
    },
    {
        "problem": "Trip distance <= 0",
        "decision": "DROP / FIX / FLAG",
        "why": "Write your reason based on the observed data.",
        "rule_owner": "State who should own this rule and why."
    },
    {
        "problem": "Passenger count == 0",
        "decision": "DROP / FIX / FLAG",
        "why": "Write your reason based on the observed data.",
        "rule_owner": "State who should own this rule and why."
    },
    {
        "problem": "Exact duplicate rows",
        "decision": "DROP / FIX / FLAG",
        "why": "Write your reason based on the observed data.",
        "rule_owner": "State who should own this rule and why."
    }
])
print(decisions.to_string(index=False))

## 5. Implement `clean(df)`
This starter function demonstrates explicit, auditable rules. Review and edit the decisions before using it in your final submission. The function returns `(cleaned_df, rows_removed)` and does not modify the input DataFrame.

For the example below, the selected policy is to drop pickups outside January (including invalid timestamps), nonpositive fares, nonpositive distances, zero passenger counts, and exact duplicate rows. If your decision table chooses `FIX` or `FLAG` for any issue, change the code accordingly and explain the difference in your report.

In [ ]:
def clean(df):
    """Return a cleaned copy and a dict of rows removed by each rule."""
    cleaned = df.copy(deep=True)
    rows_removed = {}

    initial_rows = len(cleaned)

    pickup = pd.to_datetime(cleaned[PICKUP_COL], errors="coerce")
    mask = pickup.isna() | (pickup < pd.Timestamp("2021-01-01")) | (pickup >= pd.Timestamp("2021-02-01"))
    before = len(cleaned)
    cleaned = cleaned.loc[~mask].copy()
    rows_removed["pickup_outside_january_2021"] = before - len(cleaned)

    fare = pd.to_numeric(cleaned[FARE_COL], errors="coerce")
    mask = fare <= 0
    before = len(cleaned)
    cleaned = cleaned.loc[~mask].copy()
    rows_removed["fare_nonpositive"] = before - len(cleaned)

    distance = pd.to_numeric(cleaned[DISTANCE_COL], errors="coerce")
    mask = distance <= 0
    before = len(cleaned)
    cleaned = cleaned.loc[~mask].copy()
    rows_removed["distance_nonpositive"] = before - len(cleaned)

    passenger = pd.to_numeric(cleaned[PASSENGER_COL], errors="coerce")
    mask = passenger == 0
    before = len(cleaned)
    cleaned = cleaned.loc[~mask].copy()
    rows_removed["passenger_count_zero"] = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned.drop_duplicates().copy()
    rows_removed["duplicate_rows"] = before - len(cleaned)

    rows_removed["total_rows_removed"] = initial_rows - len(cleaned)
    return cleaned, rows_removed

### Test the function on a manageable sample first
The assignment's `clean(df)` function should be tested and its output verified. Do not load the entire dataset into memory just to call this test. The following uses the first 100,000 rows as a test sample; final profiling/problem counts above cover the whole file.

In [ ]:
sample = pd.read_csv(DATA_PATH, nrows=CHUNK_SIZE, low_memory=False)
sample_before = len(sample)
sample_cleaned, sample_removed = clean(sample)

print("Rows before:", sample_before)
print("Rows after:", len(sample_cleaned))
print("Rows removed by rule:")
print(pd.Series(sample_removed).to_string())

# Verification checks
assert len(sample_cleaned) <= sample_before
assert sample.equals(pd.read_csv(DATA_PATH, nrows=CHUNK_SIZE, low_memory=False)), "Original sample was modified."
print("\\nVerification checks passed for the sample.")

## 6. Report evidence checklist
Before submitting, make sure you have:
- Printed output of `df.head()` and `df.dtypes`.
- Printed full-file profiling table.
- Printed issue-count table.
- Printed decision table with your completed explanations.
- Printed rows-before/rows-after and removal counts from `clean(df)`.
- Screenshots from class showing the actual commands and outputs. Do not recreate these screenshots if the assignment says they must be taken in class.
- Your own explanation of the pipeline flow and answers to Q1–Q3.
- GitHub repository link.

**Do not load cleaned data into PostgreSQL.**